### Dataset Provenance & Inventory Audit

In [2]:
#inspecting all parameter files
import pandas as pd
#accounts - parameter file
accounts=pd.read_csv(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\paramFiles\accounts.csv')
print('Shape:',accounts.shape)
print('Columns:',accounts.columns.to_list())
#alertPatterns - parameter file
alert_pattern=pd.read_csv(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\paramFiles\alertPatterns.csv')
print('Shape:',alert_pattern.shape)
print('Columns:',alert_pattern.columns.to_list())
#degree - parameter file
degree=pd.read_csv(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\paramFiles\degree.csv')
print('Shape:',degree.shape)
print('Columns:',degree.columns.to_list())
#transaction type - parameter file
tx_type=pd.read_csv(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\paramFiles\transactionType.csv')
print('Shape:',tx_type.shape)
print('Columns:',tx_type.columns.to_list())

Shape: (50, 6)
Columns: ['count', 'min_balance', 'max_balance', 'country', 'business_type', 'model']
Shape: (2, 15)
Columns: ['count', 'type', 'schedule_id', 'accounts', 'individual_amount', 'aggregated_amount', 'transaction_count', 'amount_difference', 'period', 'amount_rounded', 'orig_country', 'bene_country', 'orig_business', 'bene_business', 'is_fraud']
Shape: (62, 3)
Columns: ['Count', 'In-degree', 'Out-degree']
Shape: (1, 2)
Columns: ['Type', 'Frequency']


In [3]:
#inspecting the stat directory 
from pathlib import Path
folder=Path(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\stat')
for file in folder.iterdir():
    if file.is_file():
        size=file.stat().st_size
        print(file.name,size,'bytes')

alert.png 17857 bytes
cc.png 26681 bytes
count.png 31002 bytes
deg.png 26517 bytes
diameter.png 27066 bytes
wcc.png 11387 bytes


In [4]:
#checking if schema.json exist
from pathlib import Path
dataset_folder = Path(r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges")
try:

    if not dataset_folder.exists():
        raise FileNotFoundError("Dataset folder does not exist!")

    if not dataset_folder.is_dir():
        raise NotADirectoryError("Specified path is not a directory!")

    schema_files = list(dataset_folder.rglob("schema.json"))

    if not schema_files:
        raise FileNotFoundError("schema.json not found!")

    for file in schema_files:
        print("Found:", file.resolve())

except (FileNotFoundError, NotADirectoryError, PermissionError, OSError) as error:

    print(f"Error: {error}")

Error: schema.json not found!


#### AMLSim Simulation Configuration Analysis

In [5]:
import pandas as pd
from pathlib import Path
parameter_folder=Path(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\paramFiles')
csv_files=list(parameter_folder.glob("*.csv"))
for file in csv_files:
    print(file.name)
    df=pd.read_csv(file)
    print('Data Tpes',df.dtypes)
    # complete data
    print("\nComplete Data")
    print(df.to_string(index=False))


accounts.csv
Data Tpes count             int64
min_balance       int64
max_balance       int64
country          object
business_type    object
model             int64
dtype: object

Complete Data
 count  min_balance  max_balance country business_type  model
  2000          100          200      US             I      1
  2000          200          400      US             I      2
  2000          400          600      US             I      3
  2000          100          200      US             I      4
  2000          200          400      US             I      5
  2000          100          200      US             I      1
  2000          200          400      US             I      2
  2000          400          600      US             I      3
  2000          100          200      US             I      4
  2000          200          400      US             I      5
  2000          100          200      US             I      1
  2000          200          400      US             I      

##### We want to compare the account behavior models specified in paramFiles/accounts.csv against TX_BEHAVIOR_ID in our main accounts.csv

In [6]:
#step-1: Checking accounts - parameteFile
expected=(
    accounts.groupby('model')['count'].sum().sort_index()
)
print('Expected Account Distribution:',expected)

Expected Account Distribution: model
1    20000
2    20000
3    20000
4    20000
5    20000
Name: count, dtype: int64


In [7]:
#step-2: checking actual accounts - the main file
accounts_main = pd.read_csv(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\accounts.csv')

actual = (
    accounts_main
    .groupby("TX_BEHAVIOR_ID")
    .size()
    .sort_index()
)
print("Actual account distribution:")
print(actual)

Actual account distribution:
TX_BEHAVIOR_ID
1    20000
2    20000
3    20000
4    20000
5    20000
dtype: int64


In [8]:
#step-3: Comparing them
comparison = pd.concat(
    [
        expected.rename("Expected"),
        actual.rename("Actual")
    ],
    axis=1
)

comparison["Difference"] = (
    comparison["Actual"] -
    comparison["Expected"]
)

print(comparison)

   Expected  Actual  Difference
1     20000   20000           0
2     20000   20000           0
3     20000   20000           0
4     20000   20000           0
5     20000   20000           0


In [9]:
#Step-4: Validate Account Information
print("Expected accounts:", expected.sum())
print("Actual accounts:", len(accounts))

print(
    "Unique account IDs:",
    accounts_main["ACCOUNT_ID"].nunique()
)

Expected accounts: 100000
Actual accounts: 50
Unique account IDs: 100000


##### Validate account identifiers and transaction relationships

In [12]:
#Step 1 — Check account(main file) primary key integrity
accounts_main=pd.read_csv(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\accounts.csv')
print('Total Accounts:',len(accounts_main))
print("Unique Account IDs:", accounts_main['ACCOUNT_ID'].nunique())
print('Duplicate Account IDs:',accounts_main['ACCOUNT_ID'].duplicated().sum())
print('Missing Account Ids:',accounts_main['ACCOUNT_ID'].isna().sum())

Total Accounts: 100000
Unique Account IDs: 100000
Duplicate Account IDs: 0
Missing Account Ids: 0


In [19]:
#Step 2 -Check transaction foreign keys
accounts_main=pd.read_csv(r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\accounts.csv')
transactio_main_path=r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv'
valid_accounts_id=set(accounts_main['ACCOUNT_ID'].dropna())
invalid_senders=0
invalid_receivers = 0
missing_senders = 0
missing_receivers = 0
for chunk in pd.read_csv(transactio_main_path,
                         usecols=['SENDER_ACCOUNT_ID','RECEIVER_ACCOUNT_ID'],
                         chunksize=50000):
    senders = chunk["SENDER_ACCOUNT_ID"]
    receivers = chunk["RECEIVER_ACCOUNT_ID"]

    missing_senders += senders.isna().sum()
    missing_receivers += receivers.isna().sum()

    invalid_senders += (senders.notna() & ~senders.isin(valid_accounts_id)).sum()
    invalid_receivers += (receivers.notna() & ~receivers.isin(valid_accounts_id)).sum()
print("Invalid sender references:", invalid_senders)
print("Invalid receiver references:", invalid_receivers)

print("Missing sender IDs:", missing_senders)
print("Missing receiver IDs:", missing_receivers)



Invalid sender references: 0
Invalid receiver references: 0
Missing sender IDs: 0
Missing receiver IDs: 0


##### Transaction Data Quality

In [20]:
import pandas as pd


transaction_main_path=r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv'
valid_accounts_id=set(accounts_main['ACCOUNT_ID'].dropna())
invalid_senders=0
total_rows = 0
missing_values = None

unique_tx_ids = set()
duplicate_tx_ids = 0
missing_tx_ids = 0

for chunk in pd.read_csv(
    transaction_main_path,
    chunksize=500_000
):

    total_rows += len(chunk)

    # Missing values by column
    chunk_missing = chunk.isna().sum()

    if missing_values is None:
        missing_values = chunk_missing
    else:
        missing_values += chunk_missing

    # Transaction ID validation
    missing_tx_ids += chunk["TX_ID"].isna().sum()

    valid_ids = chunk["TX_ID"].dropna()

    # Detect duplicates within and across chunks
    duplicate_tx_ids += valid_ids.duplicated().sum()

    new_ids = set(valid_ids) - unique_tx_ids

    duplicate_tx_ids += len(valid_ids) - len(new_ids)

    unique_tx_ids.update(new_ids)


print("Total transactions:", total_rows)
print("Unique transaction IDs:", len(unique_tx_ids))
print("Duplicate transaction IDs:", duplicate_tx_ids)
print("Missing transaction IDs:", missing_tx_ids)

print("\nMissing values by column:")
print(missing_values)

Total transactions: 12476012
Unique transaction IDs: 12476012
Duplicate transaction IDs: 0
Missing transaction IDs: 0

Missing values by column:
TX_ID                  0
SENDER_ACCOUNT_ID      0
RECEIVER_ACCOUNT_ID    0
TX_TYPE                0
TX_AMOUNT              0
TIMESTAMP              0
IS_FRAUD               0
ALERT_ID               0
dtype: int64


##### Validate transaction amounts

In [21]:
total_amount = 0
total_transactions = 0

zero_amount_count = 0
negative_amount_count = 0

min_amount = float("inf")
max_amount = float("-inf")

for chunk in pd.read_csv(transactio_main_path,usecols=['TX_AMOUNT'],
                         chunksize=50000):
    amount=chunk['TX_AMOUNT']
    total_transactions+=len(amount)
    total_amount+=amount.sum()

    zero_amount_count+=(amount==0).sum()
    negative_amount_count+=(amount<0).sum()

    min_amount=min(min_amount,amount.min())
    max_amount=max(max_amount,amount.max())
print('Total Transactions',total_transactions)
print('Total Transactions Value', total_amount)
print('Average Transactions Value',total_amount/total_transactions)

print("Minimum amount:", min_amount)
print("Maximum amount:", max_amount)

print("Zero-value transactions:", zero_amount_count)
print("Negative-value transactions:", negative_amount_count)


Total Transactions 12476012
Total Transactions Value 282492834103.22003
Average Transactions Value 22642.879319386677
Minimum amount: 0.0
Maximum amount: 21474836.47
Zero-value transactions: 793
Negative-value transactions: 0


##### Inspect zero-value and extreme transactions

In [23]:
zero_transactions = []
largest_transactions = []
for chunk in pd.read_csv(
    transaction_main_path,
    chunksize=50000
):

    # Capture zero-value transactions
    zero_rows = chunk[chunk["TX_AMOUNT"] == 0]

    if not zero_rows.empty:
        zero_transactions.append(zero_rows)

    # Keep the largest 20 transactions from each chunk
    largest_rows = chunk.nlargest(20, "TX_AMOUNT")

    largest_transactions.append(largest_rows)


zero_df = pd.concat(zero_transactions, ignore_index=True)

largest_df = (
    pd.concat(largest_transactions, ignore_index=True)
    .nlargest(20, "TX_AMOUNT")
)

print("Zero-value transactions:")
print(zero_df.head(10).to_string(index=False))

print("\nLargest 20 transactions:")
print(largest_df.to_string(index=False))

Zero-value transactions:
 TX_ID  SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TX_TYPE  TX_AMOUNT  TIMESTAMP  IS_FRAUD  ALERT_ID
    28              63697                65992 TRANSFER        0.0          0     False        -1
    32               3712                  201 TRANSFER        0.0          0     False        -1
    36              83532                81962 TRANSFER        0.0          0     False        -1
   139              32511                10865 TRANSFER        0.0          0     False        -1
   140              13969                94787 TRANSFER        0.0          0     False        -1
   142              82382                55981 TRANSFER        0.0          0     False        -1
   211              42556                74626 TRANSFER        0.0          0     False        -1
   241              53206                 1633 TRANSFER        0.0          0     False        -1
   277              22892                93994 TRANSFER        0.0          0     False      

In [24]:
#inspect all 793 zero-value records
zero_df.groupby(
    ["TX_TYPE", "IS_FRAUD", "ALERT_ID"],
    dropna=False
).agg(
    transaction_count=("TX_ID", "count"),
    unique_senders=("SENDER_ACCOUNT_ID", "nunique"),
    unique_receivers=("RECEIVER_ACCOUNT_ID", "nunique"),
    min_timestamp=("TIMESTAMP", "min"),
    max_timestamp=("TIMESTAMP", "max")
).reset_index()

,TX_TYPE,IS_FRAUD,ALERT_ID,transaction_count,unique_senders,unique_receivers,min_timestamp,max_timestamp
0,TRANSFER,False,-1,793,489,529,0,196


In [25]:
#determine whether the maximum is a hard cap
# Frequency of the 20 largest distinct transaction amounts

largest_amounts = []

for chunk in pd.read_csv(
    transaction_main_path,
    usecols=["TX_AMOUNT"],
    chunksize=50000
):
    largest_amounts.append(
        chunk["TX_AMOUNT"].value_counts()
    )

amount_frequency = (
    pd.concat(largest_amounts)
    .groupby(level=0)
    .sum()
    .sort_index(ascending=False)
)

print(amount_frequency.head(20))

TX_AMOUNT
21474836.47    966
21370134.00      1
21360982.00      1
21349852.00      1
21338010.00      1
21241646.00      1
21237328.00      1
21234826.00      1
21215538.00      1
21211034.00      1
21178088.00      1
21171036.00      1
21160404.00      1
21149568.00      1
21147592.00      1
21144328.00      1
21140792.00      1
21133910.00      2
21127270.00      1
21122162.00      1
Name: count, dtype: int64


In [28]:
sender_ids = [42003, 3027, 93766, 92139]

print(
    accounts_main[accounts_main["ACCOUNT_ID"].isin(sender_ids)]
    .to_string(index=False)
)

 ACCOUNT_ID CUSTOMER_ID  INIT_BALANCE COUNTRY ACCOUNT_TYPE  IS_FRAUD  TX_BEHAVIOR_ID
       3027      C_3027        205.40      US            I     False               2
      42003     C_42003        239.14      US            I     False               2
      92139     C_92139        390.13      US            I      True               2
      93766     C_93766        307.94      US            I     False               2


In [30]:
max_amount = 21474836.47

max_amount_records = []

for chunk in pd.read_csv(
    transaction_main_path,
    chunksize=500_000
):

    selected = chunk[
        chunk["TX_AMOUNT"] == max_amount
    ]

    if not selected.empty:
        max_amount_records.append(selected)

max_amount_df = pd.concat(
    max_amount_records,
    ignore_index=True
)

print("Total maximum-amount transactions:",
      len(max_amount_df))

print("\nTransaction fraud labels:")
print(max_amount_df["IS_FRAUD"].value_counts(dropna=False))

print("\nUnique senders:",
      max_amount_df["SENDER_ACCOUNT_ID"].nunique())

print("Unique receivers:",
      max_amount_df["RECEIVER_ACCOUNT_ID"].nunique())

print("\nSender account fraud labels:")

print(
    accounts_main[
        accounts_main["ACCOUNT_ID"].isin(
            max_amount_df["SENDER_ACCOUNT_ID"]
        )
    ]["IS_FRAUD"].value_counts(dropna=False)
)

print("\nTransaction timestamp range:")
print(max_amount_df["TIMESTAMP"].agg(["min", "max"]))

Total maximum-amount transactions: 966

Transaction fraud labels:
IS_FRAUD
False    966
Name: count, dtype: int64

Unique senders: 380
Unique receivers: 904

Sender account fraud labels:
IS_FRAUD
False    315
True      65
Name: count, dtype: int64

Transaction timestamp range:
min     76
max    199
Name: TIMESTAMP, dtype: int64


### Timestamp Validation

In [31]:


total_rows = 0

min_timestamp = float("inf")
max_timestamp = float("-inf")

unique_timestamps = set()

timestamp_counts = {}

previous_timestamp = None
timestamp_order_violations = 0

for chunk in pd.read_csv(
    transaction_main_path,
    usecols=["TX_ID", "TIMESTAMP"],
    chunksize=500_000
):

    total_rows += len(chunk)

    timestamps = chunk["TIMESTAMP"]

    min_timestamp = min(
        min_timestamp,
        timestamps.min()
    )

    max_timestamp = max(
        max_timestamp,
        timestamps.max()
    )

    unique_timestamps.update(timestamps.unique())

    # Count transactions per timestamp
    counts = timestamps.value_counts()

    for timestamp, count in counts.items():
        timestamp_counts[timestamp] = (
            timestamp_counts.get(timestamp, 0) + count
        )

    # Check ordering across and within chunks
    if previous_timestamp is not None:
        if timestamps.iloc[0] < previous_timestamp:
            timestamp_order_violations += 1

    timestamp_order_violations += (
        timestamps.diff().dropna() < 0
    ).sum()

    previous_timestamp = timestamps.iloc[-1]


print("Total transactions:", total_rows)
print("Minimum timestamp:", min_timestamp)
print("Maximum timestamp:", max_timestamp)

print("Unique timestamps:", len(unique_timestamps))

print("Timestamp ordering violations:",
      timestamp_order_violations)

print("\nTop 10 timestamps by transaction count:")

print(
    pd.Series(timestamp_counts)
    .sort_values(ascending=False)
    .head(10)
)

Total transactions: 12476012
Minimum timestamp: 0
Maximum timestamp: 199
Unique timestamps: 200
Timestamp ordering violations: 0

Top 10 timestamps by transaction count:
130    64933
194    64515
174    64074
31     63977
100    63976
34     63903
15     63760
85     63719
45     63531
81     63527
dtype: int64


In [33]:
# Store total transaction count for each timestamp
timestamp_counts = {}

# Read file in chunks
for chunk in pd.read_csv(
    transaction_main_path,
    usecols=["TIMESTAMP"],
    chunksize=50000
):

    # Count timestamps in current chunk
    counts = chunk["TIMESTAMP"].value_counts()

    # Add counts to overall dictionary
    for timestamp, count in counts.items():

        if timestamp in timestamp_counts:
            timestamp_counts[timestamp] += count
        else:
            timestamp_counts[timestamp] = count

# Convert dictionary to Series
timestamp_series = pd.Series(timestamp_counts)

# Statistics
mean_volume = timestamp_series.mean()
median_volume = timestamp_series.median()
min_volume = timestamp_series.min()
max_volume = timestamp_series.max()
std_volume = timestamp_series.std()

# Timestamps with min/max transactions
min_timestamp = timestamp_series.idxmin()
max_timestamp = timestamp_series.idxmax()

# Transaction count at specific timestamps
count_at_0 = timestamp_counts.get(0, 0)
count_at_199 = timestamp_counts.get(199, 0)

# Print summary
print("\n===== TRANSACTION VOLUME SUMMARY =====")

print(f"Mean transaction volume     : {mean_volume:.2f}")
print(f"Median transaction volume   : {median_volume:.2f}")
print(f"Minimum transaction volume  : {min_volume}")
print(f"Maximum transaction volume  : {max_volume}")
print(f"Standard deviation          : {std_volume:.2f}")

print("\nTimestamp with MIN volume :", min_timestamp)
print("Timestamp with MAX volume :", max_timestamp)

print("\nTransactions at timestamp 0   :", count_at_0)
print("Transactions at timestamp 199 :", count_at_199)


===== TRANSACTION VOLUME SUMMARY =====
Mean transaction volume     : 62380.06
Median transaction volume   : 62399.50
Minimum transaction volume  : 60639
Maximum transaction volume  : 64933
Standard deviation          : 715.79

Timestamp with MIN volume : 158
Timestamp with MAX volume : 130

Transactions at timestamp 0   : 62487
Transactions at timestamp 199 : 63489


#### Transaction fraud labels and alert references

In [37]:
# --------------------------
# Counters
# --------------------------
total_transactions = 0

tx_type_counts = {}
alert_id_counts = {}

fraud_count = 0
non_fraud_count = 0
missing_fraud_count = 0

alert_minus1_count = 0
alert_valid_count = 0

# Relationship counters
fraud_true_alert_minus1 = 0
fraud_true_alert_valid = 0
fraud_false_alert_minus1 = 0
fraud_false_alert_valid = 0

# --------------------------
# Process Chunk-by-Chunk
# --------------------------
for chunk in pd.read_csv(
    transaction_main_path,
    usecols=["TX_TYPE", "IS_FRAUD", "ALERT_ID"],
    chunksize=50000
):

    for tx_type, is_fraud, alert_id in zip(
        chunk["TX_TYPE"],
        chunk["IS_FRAUD"],
        chunk["ALERT_ID"]
    ):

        total_transactions += 1

        # TX_TYPE count
        if tx_type not in tx_type_counts:
            tx_type_counts[tx_type] = 0
        tx_type_counts[tx_type] += 1

        # ALERT_ID count
        if alert_id not in alert_id_counts:
            alert_id_counts[alert_id] = 0
        alert_id_counts[alert_id] += 1

        # Fraud count
        if pd.isna(is_fraud):
            missing_fraud_count += 1

        elif is_fraud:
            fraud_count += 1

            if alert_id == -1:
                fraud_true_alert_minus1 += 1
            else:
                fraud_true_alert_valid += 1

        else:
            non_fraud_count += 1

            if alert_id == -1:
                fraud_false_alert_minus1 += 1
            else:
                fraud_false_alert_valid += 1

        # Alert count
        if alert_id == -1:
            alert_minus1_count += 1
        else:
            alert_valid_count += 1

# --------------------------
# Final Metrics
# --------------------------
fraud_percentage = (
    fraud_count / total_transactions * 100
    if total_transactions > 0
    else 0
)

# --------------------------
# Summary
# --------------------------
print("\n========== SUMMARY ==========")

print(f"Total Transactions: {total_transactions:,}")

print("\nFraud Labels")
print(f"Fraud            : {fraud_count:,}")
print(f"Non Fraud        : {non_fraud_count:,}")
print(f"Missing Labels   : {missing_fraud_count:,}")
print(f"Fraud Percentage : {fraud_percentage:.2f}%")

print("\nAlert References")
print(f"ALERT_ID = -1  : {alert_minus1_count:,}")
print(f"ALERT_ID != -1 : {alert_valid_count:,}")

print("\nFraud vs Alert")
print(f"Fraud=True  & Alert=-1  : {fraud_true_alert_minus1:,}")
print(f"Fraud=True  & Alert!=-1 : {fraud_true_alert_valid:,}")
print(f"Fraud=False & Alert=-1  : {fraud_false_alert_minus1:,}")
print(f"Fraud=False & Alert!=-1 : {fraud_false_alert_valid:,}")

print("\nTransaction Types")
for tx_type, count in sorted(
    tx_type_counts.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{tx_type}: {count:,}")


========== SUMMARY ==========
Total Transactions: 12,476,012

Fraud Labels
Fraud            : 17,052
Non Fraud        : 12,458,960
Missing Labels   : 0
Fraud Percentage : 0.14%

Alert References
ALERT_ID = -1  : 12,458,960
ALERT_ID != -1 : 17,052

Fraud vs Alert
Fraud=True  & Alert=-1  : 0
Fraud=True  & Alert!=-1 : 17,052
Fraud=False & Alert=-1  : 12,458,960
Fraud=False & Alert!=-1 : 0

Transaction Types
TRANSFER: 12,476,012


##### Finding 1: Severe class imbalance:Only approximately 0.14% of transactions are fraud-labelled.This means a model that predicts every transaction as non-fraud would achieve approximately 99.86% accuracy while detecting zero fraud-labelled transactions.

##### Finding 2: Fraud labels and alert references are perfectly aligned. In this dataset, every transaction labelled IS_FRAUD=True has a non-negative alert reference, while every non-fraud-labelled transaction has ALERT_ID=-1.

##### Finding 3: Alert references appear to identify the labelled transactions. The matching count of 17,052 fraud-labelled transactions and 17,052 transactions with non-negative alert IDs is encouraging.

### Validate alerts.csv - main file

In [44]:
import pandas as pd

# ==================================================
# FILE PATHS
# ==================================================

alert_main = r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv"

# Assuming already defined
# transaction_main_path = r"...\transactions.csv"

# ==================================================
# STEP 1: Inspect alerts.csv
# ==================================================

print("\n========== ALERTS FILE INSPECTION ==========")

alerts_sample = pd.read_csv(alert_main, nrows=1000)

print("\nColumns:")
print(list(alerts_sample.columns))

print("\nData Types:")
print(alerts_sample.dtypes)

missing_counts = pd.Series(
    0,
    index=alerts_sample.columns,
    dtype="int64",
)

for chunk in pd.read_csv(alert_main, chunksize=50_000):
    missing_counts += chunk.isna().sum()

print("\nMissing values across the complete alerts file:")
print(missing_counts)

# ==================================================
# STEP 2: Build transaction lookup
# ==================================================

print("\nBuilding transaction lookup...")

transaction_lookup = {}

for chunk in pd.read_csv(
    transaction_main_path,
    usecols=[
        "TX_ID",
        "SENDER_ACCOUNT_ID",
        "RECEIVER_ACCOUNT_ID",
        "IS_FRAUD"
    ],
    chunksize=50000
):

    for tx_id, sender, receiver, fraud in zip(
        chunk["TX_ID"],
        chunk["SENDER_ACCOUNT_ID"],
        chunk["RECEIVER_ACCOUNT_ID"],
        chunk["IS_FRAUD"]
    ):

        transaction_lookup[tx_id] = (
            sender,
            receiver,
            fraud
        )

print(f"Transactions Loaded: {len(transaction_lookup):,}")

# ==================================================
# STEP 3: Validation Counters
# ==================================================

total_alerts = 0
missing_alert_id = 0

alert_ids_seen = set()
duplicate_alert_ids = 0

tx_ids_seen = set()
duplicate_tx_ids = 0

existing_tx_refs = 0
missing_tx_refs = 0

fraud_match = 0
fraud_mismatch = 0

sender_match = 0
sender_mismatch = 0

receiver_match = 0
receiver_mismatch = 0

alert_type_counts = {}
alert_type_fraud = {}
alert_type_nonfraud = {}

# ==================================================
# STEP 4: Validate alerts.csv
# ==================================================

for chunk in pd.read_csv(alert_main, chunksize=50000):

    for (
        alert_id,
        tx_id,
        alert_type,
        alert_fraud,
        alert_sender,
        alert_receiver
    ) in zip(
        chunk["ALERT_ID"],
        chunk["TX_ID"],
        chunk["ALERT_TYPE"],
        chunk["IS_FRAUD"],
        chunk["SENDER_ACCOUNT_ID"],
        chunk["RECEIVER_ACCOUNT_ID"]
    ):

        total_alerts += 1

        # -----------------------------
        # ALERT_ID validation
        # -----------------------------

        if pd.isna(alert_id):
            missing_alert_id += 1

        else:
            if alert_id in alert_ids_seen:
                duplicate_alert_ids += 1
            else:
                alert_ids_seen.add(alert_id)

        # -----------------------------
        # TX_ID validation
        # -----------------------------

        if tx_id in tx_ids_seen:
            duplicate_tx_ids += 1
        else:
            tx_ids_seen.add(tx_id)

        # -----------------------------
        # ALERT TYPE COUNTS
        # -----------------------------

        if alert_type not in alert_type_counts:
            alert_type_counts[alert_type] = 0

        alert_type_counts[alert_type] += 1

        # -----------------------------
        # Transaction lookup validation
        # -----------------------------

        if tx_id not in transaction_lookup:
            missing_tx_refs += 1
            continue

        existing_tx_refs += 1

        tx_sender, tx_receiver, tx_fraud = transaction_lookup[tx_id]

        # -----------------------------
        # Fraud label validation
        # -----------------------------

        if alert_fraud == tx_fraud:
            fraud_match += 1
        else:
            fraud_mismatch += 1

        # -----------------------------
        # Sender validation
        # -----------------------------

        if alert_sender == tx_sender:
            sender_match += 1
        else:
            sender_mismatch += 1

        # -----------------------------
        # Receiver validation
        # -----------------------------

        if alert_receiver == tx_receiver:
            receiver_match += 1
        else:
            receiver_mismatch += 1

        # -----------------------------
        # Alert Type x Fraud
        # -----------------------------

        if alert_type not in alert_type_fraud:
            alert_type_fraud[alert_type] = 0

        if alert_type not in alert_type_nonfraud:
            alert_type_nonfraud[alert_type] = 0

        if alert_fraud:
            alert_type_fraud[alert_type] += 1
        else:
            alert_type_nonfraud[alert_type] += 1

# ==================================================
# STEP 5: REPORT
# ==================================================

print("\n")
print("=" * 60)
print("ALERT VALIDATION REPORT")
print("=" * 60)

print("\n----- ALERT_ID VALIDATION -----")
print(f"Total Alert Records      : {total_alerts:,}")
print(f"Unique ALERT_ID          : {len(alert_ids_seen):,}")
print(f"Duplicate ALERT_ID       : {duplicate_alert_ids:,}")
print(f"Missing ALERT_ID         : {missing_alert_id:,}")

print("\n----- TX_ID VALIDATION -----")
print(f"Valid TX References      : {existing_tx_refs:,}")
print(f"Missing TX References    : {missing_tx_refs:,}")
print(f"Duplicate TX_ID Alerts   : {duplicate_tx_ids:,}")

print("\n----- FRAUD LABEL CHECK -----")
print(f"Matching Labels          : {fraud_match:,}")
print(f"Mismatching Labels       : {fraud_mismatch:,}")

print("\n----- ACCOUNT VALIDATION -----")
print(f"Sender Matches           : {sender_match:,}")
print(f"Sender Mismatches        : {sender_mismatch:,}")
print(f"Receiver Matches         : {receiver_match:,}")
print(f"Receiver Mismatches      : {receiver_mismatch:,}")

print("\n----- ALERT TYPE DISTRIBUTION -----")

for alert_type in sorted(alert_type_counts):

    print(
        f"{alert_type}: "
        f"Total={alert_type_counts,}, "
        f"Fraud={alert_type_fraud.get(alert_type, 0):,}, "
        f"NonFraud={alert_type_nonfraud.get(alert_type, 0):,}"
    )

print("\n" + "=" * 60)


========== ALERTS FILE INSPECTION ==========

Columns:
['ALERT_ID', 'ALERT_TYPE', 'IS_FRAUD', 'TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP']

Data Types:
ALERT_ID                 int64
ALERT_TYPE              object
IS_FRAUD                  bool
TX_ID                    int64
SENDER_ACCOUNT_ID        int64
RECEIVER_ACCOUNT_ID      int64
TX_TYPE                 object
TX_AMOUNT              float64
TIMESTAMP                int64
dtype: object

Missing values across the complete alerts file:
ALERT_ID               0
ALERT_TYPE             0
IS_FRAUD               0
TX_ID                  0
SENDER_ACCOUNT_ID      0
RECEIVER_ACCOUNT_ID    0
TX_TYPE                0
TX_AMOUNT              0
TIMESTAMP              0
dtype: int64

Building transaction lookup...
Transactions Loaded: 12,476,012


ALERT VALIDATION REPORT

----- ALERT_ID VALIDATION -----
Total Alert Records      : 17,052
Unique ALERT_ID          : 3,910
Duplicate ALERT_ID       : 13,14

#### Investtigate ALERT_ID structure


In [46]:
import pandas as pd
from statistics import median

alert_main = r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv"
alert_record_counts = {}      
alert_types = {}           
alert_fraud_labels = {}      
alert_tx_ids = {}            

tx_to_alerts = {}            

# PROCESS ALERTS FILE
for chunk in pd.read_csv(alert_main, chunksize=50000):

    for alert_id, tx_id, alert_type, fraud_label in zip(
        chunk["ALERT_ID"],
        chunk["TX_ID"],
        chunk["ALERT_TYPE"],
        chunk["IS_FRAUD"]
    ):

       
        # ALERT_ID -> Record Count
        

        if alert_id not in alert_record_counts:
            alert_record_counts[alert_id] = 0

        alert_record_counts[alert_id] += 1

        
        # ALERT_ID -> ALERT_TYPE
        

        if alert_id not in alert_types:
            alert_types[alert_id] = set()

        alert_types[alert_id].add(alert_type)

        
        # ALERT_ID -> FRAUD LABELS
        

        if alert_id not in alert_fraud_labels:
            alert_fraud_labels[alert_id] = set()

        alert_fraud_labels[alert_id].add(fraud_label)

        
        # ALERT_ID -> TX_ID
        

        if alert_id not in alert_tx_ids:
            alert_tx_ids[alert_id] = set()

        alert_tx_ids[alert_id].add(tx_id)

        
        # TX_ID -> ALERT_ID
        

        if tx_id not in tx_to_alerts:
            tx_to_alerts[tx_id] = set()

        tx_to_alerts[tx_id].add(alert_id)


# ALERT GROUP SIZE STATISTICS


group_sizes = list(alert_record_counts.values())

num_alert_ids = len(group_sizes)

min_group_size = min(group_sizes)
max_group_size = max(group_sizes)

avg_group_size = sum(group_sizes) / len(group_sizes)

median_group_size = median(group_sizes)


# MULTIPLE ALERT TYPES


multi_type_alerts = 0

for alert_id, types in alert_types.items():

    if len(types) > 1:
        multi_type_alerts += 1


# INCONSISTENT FRAUD LABELS


inconsistent_fraud_alerts = 0

for alert_id, frauds in alert_fraud_labels.items():

    if len(frauds) > 1:
        inconsistent_fraud_alerts += 1


# TX_ID COUNTS PER ALERT


distinct_tx_per_alert = {}

for alert_id, tx_set in alert_tx_ids.items():

    distinct_tx_per_alert[alert_id] = len(tx_set)


# TX_ID BELONGING TO MULTIPLE ALERTS


shared_tx_count = 0

for tx_id, alert_set in tx_to_alerts.items():

    if len(alert_set) > 1:
        shared_tx_count += 1


# TOP 10 LARGEST ALERT GROUPS


top_10_alerts = sorted(
    alert_record_counts.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

# ==================================================
# REPORT
# ==================================================

print("\n" + "=" * 70)
print("ALERT ID STRUCTURE INVESTIGATION")
print("=" * 70)

print("\n----- ALERT GROUP COUNTS -----")
print(f"Unique ALERT_IDs                  : {num_alert_ids:,}")

print("\n----- GROUP SIZE STATISTICS -----")
print(f"Minimum Records Per Alert         : {min_group_size:,}")
print(f"Maximum Records Per Alert         : {max_group_size:,}")
print(f"Average Records Per Alert         : {avg_group_size:.2f}")
print(f"Median Records Per Alert          : {median_group_size}")

print("\n----- ALERT TYPE CONSISTENCY -----")
print(f"Alert IDs With Multiple Types     : {multi_type_alerts:,}")

print("\n----- FRAUD LABEL CONSISTENCY -----")
print(f"Alert IDs With Mixed Labels       : {inconsistent_fraud_alerts:,}")

print("\n----- TRANSACTION REFERENCES -----")
print(f"TX_IDs Linked To Multiple Alerts  : {shared_tx_count:,}")

print("\n----- TOP 10 LARGEST ALERT GROUPS -----")

for alert_id, count in top_10_alerts:

    print("-" * 50)

    print(f"ALERT_ID            : {alert_id}")

    print(f"Record Count        : {count:,}")

    print(
        f"Distinct TX_IDs     : "
        f"{distinct_tx_per_alert,}"
    )

    print(
        f"Alert Types         : "
        f"{list(alert_types[alert_id])}"
    )

    print(
        f"Fraud Labels        : "
        f"{list(alert_fraud_labels[alert_id])}"
    )

print("\n" + "=" * 70)


ALERT ID STRUCTURE INVESTIGATION

----- ALERT GROUP COUNTS -----
Unique ALERT_IDs                  : 3,910

----- GROUP SIZE STATISTICS -----
Minimum Records Per Alert         : 1
Maximum Records Per Alert         : 5
Average Records Per Alert         : 4.36
Median Records Per Alert          : 4.0

----- ALERT TYPE CONSISTENCY -----
Alert IDs With Multiple Types     : 0

----- FRAUD LABEL CONSISTENCY -----
Alert IDs With Mixed Labels       : 0

----- TRANSACTION REFERENCES -----
TX_IDs Linked To Multiple Alerts  : 0

----- TOP 10 LARGEST ALERT GROUPS -----
--------------------------------------------------
ALERT_ID            : 3000
Record Count        : 5
Distinct TX_IDs     : ({2036: 3, 842: 1, 3000: 5, 280: 4, 2861: 3, 1252: 4, 1147: 2, 802: 3, 2600: 4, 1106: 4, 2786: 3, 2455: 4, 3921: 4, 3207: 1, 2601: 3, 1394: 2, 2613: 1, 2129: 3, 1382: 3, 3887: 2, 839: 2, 212: 4, 909: 1, 1399: 1, 1597: 2, 3139: 4, 2313: 2, 3239: 5, 112: 2, 3083: 2, 567: 3, 3884: 3, 1757: 1, 2697: 5, 2909: 5, 358